# Modelo de Probabilidad

In [644]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
from sklearn.preprocessing import LabelEncoder

df_qs = pd.read_pickle('ML_QS.pkl')
df_aug = pd.read_pickle('ML_Aug.pkl')
df = pd.read_pickle('ML_Aug.pkl')

In [645]:
df_qs["degree"].unique()

array(['Arts & Humanities', 'Archaeology', 'Architecture', 'Art & Design',
       'Classics', 'English Language', 'History$', 'History of Art',
       'Linguistics', 'Modern Languages', 'Music', 'Performing Arts',
       'Philosophy', 'Theology', 'Engineering & Technology',
       'Computer Science', 'Data Science', 'Engineering - Chemical',
       'Engineering - Civil', 'Engineering - Electrical',
       'Engineering - Mechanical', 'Engineering - Mineral',
       'Petroleum Engineering', 'Life Sciences & Medicine', 'Agriculture',
       'Anatomy', 'Biological', 'Dentistry', 'Medicine', 'Nursing',
       'Pharmacy', 'Psychology', 'Veterinary Science', 'Natural Sciences',
       'Chemistry', 'Earth & Marine Sciences', 'Environmental Sciences',
       'Geography', 'Geology', 'Geophysics', 'Mathematics',
       'Materials Science', 'Physics', 'Social Sciences & Management',
       'Accounting', 'Anthropology', 'Business', 'Communication',
       'Development Studies', 'Economics & Econome

## RandomForestClassifier Training

In [646]:
le_uni = LabelEncoder()
le_area = LabelEncoder()

df["university"] = le_uni.fit_transform(df["university"])
df["degree"] = le_area.fit_transform(df["degree"])

In [647]:
X = df.drop(columns=['admission', 'country'])
y = df['admission']

feature_order = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model_rForest = RandomForestClassifier()
model_rForest.fit(X_train, y_train)
print(classification_report(y_test, model_rForest.predict(X_test)))

              precision    recall  f1-score   support

           0       0.91      0.67      0.77        75
           1       0.81      0.95      0.88       111

    accuracy                           0.84       186
   macro avg       0.86      0.81      0.82       186
weighted avg       0.85      0.84      0.83       186



In [648]:
features = ['gpa', 'toefl', 'sat', 'profile', 'peak',
            'university', 'degree', 'sat_required', 'qs_score']

df["prob"] = model_rForest.predict_proba(df[features])[:, 1]

### Feature Weight

In [649]:
importances = model_rForest.feature_importances_
for feat, imp in zip(features, importances):
    print(f"{feat}: {imp:.4f}")

gpa: 0.0902
toefl: 0.0946
sat: 0.0907
profile: 0.0641
peak: 0.0071
university: 0.2799
degree: 0.0841
sat_required: 0.0099
qs_score: 0.2795


## Modelo Top-K de Recomendacion

recom_score = α * prob + β * (qs_score_normalizado)

In [650]:
df_topk = pd.concat([df_aug[['university', 'country', 'degree', 'qs_score']], 
                     df_qs[['university', 'country', 'degree', 'qs_score']]], 
                    ignore_index=True).drop_duplicates(subset=['degree', 'country', 'university'])


In [651]:
def recomendar_top_k_ponderado(perfil_estudiante, country, major, df, modelo_rf, k, alpha, beta):

    df = df[
        (df["degree"] == major) &
        (df["country"] == country.upper())
    ]


    for key in ["gpa", "toefl", "sat", "profile", "peak", "sat_required"]:
        df[key] = perfil_estudiante[key]

    le_university = LabelEncoder()
    le_degree = LabelEncoder()
    df["university"] = le_university.fit_transform(df["university"])
    df["degree"] = le_degree.fit_transform(df["degree"])

    # 4. Predecir probabilidad de admisión
    features = ['gpa', 'toefl', 'sat', 'profile', 'peak',
            'university', 'degree', 'sat_required', 'qs_score']
    df["prob"] = modelo_rf.predict_proba(df[features])[:, 1]

    # 5. Normalizar qs_score
    qs_min = df["qs_score"].min()
    qs_max = df["qs_score"].max()
    df["qs_score_norm"] = (df["qs_score"] - qs_min) / (qs_max - qs_min)

    # 6. Calcular recom_score
    df["recom_score"] = alpha * df["prob"] + beta * df["qs_score_norm"]

    # 7. Decodificar labels
    df["university"] = le_university.inverse_transform(df["university"])
    df["degree"] = le_degree.inverse_transform(df["degree"])

    # 8. Top-K
    return df[["country", "university", "degree", "qs_score", "prob", "qs_score_norm", "recom_score"]]\
            .sort_values(by="recom_score", ascending=False).head(k)

## UI

In [655]:
perfil = {
        "gpa": 3.33,
        "toefl": 90,
        "sat": 1340,
        "profile": 14,
        "peak": 0,
        "sat_required": 1,
}
country = "united states"
major = "Engineering - Mechanical"
alfa = 0.8 #prob
beta = 0.2 #rank
k = 10

recomendar_top_k_ponderado(perfil, country, major, df_topk, model_rForest, k, alpha, beta)

,country,university,degree,qs_score,prob,qs_score_norm,recom_score
7571,UNITED STATES,TEXAS TECH UNIVERSITY,Engineering - Mechanical,53.2,0.798333,0.085774,0.575988
7581,UNITED STATES,UNIVERSITY AT BUFFALO SUNY,Engineering - Mechanical,52.7,0.788333,0.075314,0.566896
7641,UNITED STATES,UNIVERSITY OF ALABAMA,Engineering - Mechanical,50.3,0.798333,0.025105,0.563854
7566,UNITED STATES,OKLAHOMA STATE UNIVERSITY,Engineering - Mechanical,53.4,0.778333,0.089958,0.562825
7650,UNITED STATES,UNIVERSITY OF AKRON,Engineering - Mechanical,49.9,0.798333,0.016736,0.562181
7570,UNITED STATES,ROCHESTER INSTITUTE OF TECHNOLOGY (RIT),Engineering - Mechanical,53.2,0.778333,0.085774,0.561988
7584,UNITED STATES,UNIVERSITY OF CALIFORNIA RIVERSIDE,Engineering - Mechanical,52.6,0.778333,0.073222,0.559478
7630,UNITED STATES,OREGON STATE UNIVERSITY,Engineering - Mechanical,50.7,0.788333,0.033473,0.558528
7189,UNITED STATES,COLUMBIA UNIVERSITY,Engineering - Mechanical,73.4,0.638333,0.508368,0.548507
7248,UNITED STATES,ARIZONA STATE UNIVERSITY,Engineering - Mechanical,68.8,0.658333,0.412134,0.543260


In [620]:
print(df_topk[(df_topk['degree'] == 'Business') & 
                        (df_topk['country'] == 'UNITED STATES')].sort_values(by="qs_score", ascending=False))

                                        university        country    degree  \
17630                           HARVARD UNIVERSITY  UNITED STATES  Business   
17631  MASSACHUSETTS INSTITUTE OF TECHNOLOGY (MIT)  UNITED STATES  Business   
17632                          STANFORD UNIVERSITY  UNITED STATES  Business   
397                     UNIVERSITY OF PENNSYLVANIA  UNITED STATES  Business   
17640      UNIVERSITY OF CALIFORNIA BERKELEY (UCB)  UNITED STATES  Business   
...                                            ...            ...       ...   
733                        SACRED HEART UNIVERSITY  UNITED STATES  Business   
595                              AUBURN UNIVERSITY  UNITED STATES  Business   
541                            CSUNIVERSITY FRESNO  UNITED STATES  Business   
343                       CSUNIVERSITY LOS ANGELES  UNITED STATES  Business   
544                          DIABLO VALLEY COLLEGE  UNITED STATES  Business   

        qs_score  
17630  96.200000  
17631  92.400